# 12 · Runtime context

**Goal:** by the end you can pass run-level settings (such as how many decimal places to show) into a graph with
`context=`, read them in a node through `runtime.context`, and explain the difference between state and context.

## The idea in plain English

The **state** is the data the graph works on. It changes as nodes run: `parse` adds `a` and `b`, `calculate` adds
`result`. Some information isn't like that. "Show answers with 2 decimal places" is a **setting** for the whole run.
No node should change it, and it isn't part of the answer.

LangGraph keeps such settings apart from the state, in the **runtime context**. You describe it with a
**context schema**, pass the values when you call `invoke(..., context={...})`, and a node reads them through an
extra parameter called `runtime`. The same compiled graph can then run with different settings each time.

**Analogy:** cooking. The ingredients change as you cook: that's the state. The oven temperature you set before you
start is the context. It's read by the recipe steps, but it isn't an ingredient, and it doesn't end up on the plate.

## Picture

Context goes in next to the input. Only the node that asks for it (`format`) reads it, and it never becomes part of
the state.

```mermaid
flowchart LR
    S([START]) --> parse --> calculate --> format --> E([END])
    ctx[["context<br/>decimal_places = 2"]] -. "runtime.context" .-> format
```

## Step 1: The context schema

A **`dataclass`** (lesson 02) is the usual way to write a context schema. Its default value is documentation of a
sensible setting. (Careful: see Step 8 for when the default is *not* used.)

In [1]:
from dataclasses import dataclass


@dataclass
class Context:
    decimal_places: int = 2


print(Context())
print(Context(decimal_places=4))

Context(decimal_places=2)
Context(decimal_places=4)


## Step 2: The state and the first two nodes

The usual pipeline state. `parse` and `calculate` don't need any settings, so they look exactly as before.

In [2]:
from typing import TypedDict

from langgraph.graph import StateGraph, START, END


class CalcState(TypedDict):
    expression: str
    a: float
    op: str
    b: float
    result: float
    answer: str

`parse` and `calculate` are unchanged from lesson 04. They need no settings, so they don't take a `runtime` parameter.

In [3]:
def parse(state: CalcState) -> dict:
    left, op, right = state["expression"].split()
    return {"a": float(left), "op": op, "b": float(right)}


def calculate(state: CalcState) -> dict:
    a, op, b = state["a"], state["op"], state["b"]
    if op == "+":
        return {"result": a + b}
    elif op == "-":
        return {"result": a - b}
    elif op == "*":
        return {"result": a * b}
    return {"result": a / b}

## Step 3: A node that reads the context

**`Runtime`** (from `langgraph.runtime`) is an object LangGraph passes to a node that asks for it. You ask by adding a
second parameter named exactly `runtime`, with the type hint `Runtime[Context]`. The settings are in
`runtime.context`.

In the f-string, `{value:.{places}f}` means "show `value` with `places` digits after the decimal point". For example,
`f"{3.14159:.2f}"` is `"3.14"`.

In [4]:
from langgraph.runtime import Runtime


def format_answer(state: CalcState, runtime: Runtime[Context]) -> dict:
    places = runtime.context.decimal_places
    return {"answer": f"{state['expression']} = {state['result']:.{places}f}"}

## Step 4: Build the graph with `context_schema`

The context schema is the second argument of `StateGraph`, `context_schema=Context`. The wiring is the lesson 04
pipeline. The context doesn't appear in the diagram, because it isn't a node or an edge.

In [5]:
builder = StateGraph(CalcState, context_schema=Context)
builder.add_node("parse", parse)
builder.add_node("calculate", calculate)
builder.add_node("format", format_answer)
builder.add_edge(START, "parse")
builder.add_edge("parse", "calculate")
builder.add_edge("calculate", "format")
builder.add_edge("format", END)

graph = builder.compile()
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	calculate(calculate)
	format(format)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	calculate --> format;
	parse --> calculate;
	format --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Step 5: The same graph, different settings

`context=` is a keyword argument of `invoke`, next to the input. A plain dict is fine: LangGraph turns it into a
`Context` object for you.

In [6]:
for places in [0, 2, 5]:
    final = graph.invoke({"expression": "10 / 3"}, context={"decimal_places": places})
    print(f"decimal_places={places}:", final["answer"])

decimal_places=0: 10 / 3 = 3
decimal_places=2: 10 / 3 = 3.33
decimal_places=5: 10 / 3 = 3.33333


You can also pass a `Context` object directly. It's the same thing.

In [7]:
final = graph.invoke({"expression": "22 / 7"}, context=Context(decimal_places=4))
print(final["answer"])

22 / 7 = 3.1429


## Step 6: The context is not part of the state

Look at the final state: there's no `decimal_places` key. The context was read during the run, but it isn't data
the graph produced, so it isn't returned.

In [8]:
print(final)
print("decimal_places in state?", "decimal_places" in final)

{'expression': '22 / 7', 'a': 22.0, 'op': '/', 'b': 7.0, 'result': 3.142857142857143, 'answer': '22 / 7 = 3.1429'}
decimal_places in state? False


## Step 7: What `runtime` holds

`runtime.context` is the settings object. `Runtime` holds other run-level things too. One of them, `runtime.store`,
is the long-term memory you'll meet in lesson 17. It's `None` here because we didn't give the graph a store.

In [9]:
def show_runtime(state: CalcState, runtime: Runtime[Context]) -> dict:
    print("  runtime.context:", runtime.context)
    print("  runtime.store:  ", runtime.store)
    return {}


builder = StateGraph(CalcState, context_schema=Context)
builder.add_node("show_runtime", show_runtime)
builder.add_edge(START, "show_runtime")
builder.add_edge("show_runtime", END)
builder.compile().invoke({"expression": "1 + 1"}, context={"decimal_places": 3});

  runtime.context: Context(decimal_places=3)
  runtime.store:   None


## Step 8: Forgetting the context

If you call `invoke` with **no** `context=` at all, `runtime.context` is `None`. The dataclass default is *not* used,
because no `Context` object was ever created. The node then fails.

In [10]:
try:
    graph.invoke({"expression": "10 / 3"})
except AttributeError as err:
    print("AttributeError:", err)

AttributeError: 'NoneType' object has no attribute 'decimal_places'


If a setting should have a default, make the node fall back to it. `runtime.context or Context()` means "use the
context if there is one, otherwise a `Context` with its default values".

In [11]:
def format_with_default(state: CalcState, runtime: Runtime[Context]) -> dict:
    context = runtime.context or Context()  # fall back to the defaults
    return {"answer": f"{state['expression']} = {state['result']:.{context.decimal_places}f}"}


builder = StateGraph(CalcState, context_schema=Context)
builder.add_node("parse", parse)
builder.add_node("calculate", calculate)
builder.add_node("format", format_with_default)
builder.add_edge(START, "parse")
builder.add_edge("parse", "calculate")
builder.add_edge("calculate", "format")
builder.add_edge("format", END)
forgiving = builder.compile()

print(forgiving.invoke({"expression": "10 / 3"})["answer"])
print(forgiving.invoke({"expression": "10 / 3"}, context={"decimal_places": 1})["answer"])

10 / 3 = 3.33
10 / 3 = 3.3


## State or context?

| | State | Context |
|---|---|---|
| What it holds | Data the run works on and produces | Settings for the whole run |
| Can nodes change it? | Yes, by returning updates | No, it's fixed when you call `invoke` |
| How you pass it | The first argument of `invoke` | `invoke(..., context={...})` |
| How a node reads it | `state["key"]` | `runtime.context.key` |
| Returned by `invoke`? | Yes | No |
| Examples | `expression`, `result`, `answer`, `history` | decimal places, user ID, language, which model to use |

## What just happened

- **Step 1** made a `Context` dataclass with one setting, `decimal_places`, defaulting to `2`.
- **Step 3** wrote `format_answer(state, runtime: Runtime[Context])`, which reads `runtime.context.decimal_places`.
- **Step 4** passed `context_schema=Context` to `StateGraph`. The diagram is the same three-node pipeline.
- **Step 5** ran one compiled graph with three settings: `10 / 3 = 3`, `10 / 3 = 3.33` and `10 / 3 = 3.33333`.
- **Step 6** showed the final state has no `decimal_places` key. Context is never returned.
- **Step 7** printed `runtime.context` as `Context(decimal_places=3)`, and `runtime.store` as `None`.
- **Step 8** showed that with no `context=`, `runtime.context` is `None`, and fixed it with
  `runtime.context or Context()`.

## Common mistakes

1. **Giving the `runtime` parameter another name.** LangGraph finds it by its name. `def format_answer(state, rt:
   Runtime[Context])` fails when the graph runs:
   `TypeError: format_answer() missing 1 required positional argument: 'rt'`.
   Fix: call the parameter `runtime`.
2. **Calling `invoke` without `context=`.** `runtime.context` is `None`, so reading a setting fails:
   `AttributeError: 'NoneType' object has no attribute 'decimal_places'`.
   Fix: always pass `context=`, or fall back with `runtime.context or Context()` (Step 8).
3. **A misspelled setting name.** `context={"decimals": 4}` fails when LangGraph builds the `Context`:
   `TypeError: Context.__init__() got an unexpected keyword argument 'decimals'`.
   Fix: use the field names of your context dataclass.

## Try it

1. Add a second setting, `show_expression: bool = True`. When it's `False`, the answer should be just the number,
   such as `3.33`. Run `"10 / 3"` both ways.
2. For each of these, decide whether it belongs in the **state** or the **context**, and why: the expression the user
   typed; the user's preferred language; the running total of a session; the maximum number of retries; the
   `result` of a calculation.

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **Runtime context**: settings fixed for one run, passed with `invoke(..., context={...})` and not part of the state.
- **Context schema**: the class (usually a `dataclass`) describing the context; passed as `context_schema=`.
- **`Runtime`**: the object LangGraph passes to a node's `runtime` parameter. It holds `context`, `store` and more.